<a href="https://colab.research.google.com/github/ml4devs/ml4devs-notebooks/blob/master/llm/openai_responses_api_vs_chat_completions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1><center>OpenAI Responses API vs Chat Completions: What Changed</center></h1>

<p><center>
<address>&copy; Satish Chandra Gupta<br/>
LinkedIn: <a href="https://www.linkedin.com/in/scgupta/">scgupta</a>,
Twitter: <a href="https://twitter.com/scgupta">scgupta</a>
</address>
</center></p>

---

Blog Post: [OpenAI Responses API vs Chat Completions: What Changed](https://www.ml4devs.com/articles/openai-responses-api-vs-chat-completions/)

## Introduction

OpenAI shipped [`/v1/responses`](https://developers.openai.com/api/docs/api-reference/responses) in March 2025 and called it the new API primitive. Since then it has picked up the features that only exist there: hosted conversation state, built-in tools that run on OpenAI's side, background execution, and reasoning that survives across turns. The older [`/v1/chat/completions`](https://developers.openai.com/api/docs/api-reference/chat) is not deprecated and OpenAI has said it stays supported. The [Assistants API](https://developers.openai.com/api/docs/deprecations) is the one being retired, with a sunset date of 26 August 2026, and Responses is its replacement.

So this is not a forced migration. It is a choice, and the choice has a cost on both sides.

This notebook runs both APIs side by side against the same model and shows the differences as executed code rather than as a table you have to trust:

- **Request shape.** `input` and `instructions` in place of a `messages` array.
- **Response shape.** An item list in `output` in place of `choices[].message`.
- **Conversation state.** `previous_response_id` and the Conversations API in place of resending the whole array, and what that does to your token bill.
- **Tools.** Built-in tools such as web search as one line of config, plus the smaller changes to how you declare your own functions.
- **Reasoning.** How reasoning tokens are carried from turn to turn, stateful and stateless.
- **Parameter parity.** What Chat Completions has that Responses does not, computed live from the SDK rather than quoted.
- **The decision rule.** When porting pays for itself, and when staying put is the better engineering call.

### Who This Is For

You have called `chat.completions.create()` before and you are deciding whether to port. If you have never made an LLM API call, start with [Build Your First LLM Application with the OpenAI API](https://www.ml4devs.com/articles/llm-first-app-with-openai-api/) and come back.

### What This Costs to Run

Every cell below makes real API calls. Using `gpt-5.4-nano` at $0.20 per million input tokens and $1.25 per million output tokens, the text calls come to a couple of cents in total. The built-in web search demos cost more than all of them combined, because hosted tools are billed per call at $10 per 1,000 calls and one request can trigger several searches. Budget under ten cents for a full run, and skip the two web search cells if you want it closer to two.

---

## Setup Environment

### Install Pip Packages

You need Python 3.8 or higher and a recent [OpenAI Python API library](https://github.com/openai/openai-python). Responses API support arrived in version 1.66.0, released the day the API shipped, and several things used below were added well after that, so install the latest rather than pinning to that floor.

In [1]:
# You should have Python 3.8 or higher

!python --version

Python 3.11.1


In [2]:
!pip install --upgrade openai python-dotenv &> /dev/null

In [3]:
import openai

print(f"openai SDK version: {openai.__version__}")

openai SDK version: 3.0.0


### Upload `.env` File with API Keys

You can either use OpenAI directly, or Azure OpenAI from Microsoft. Both support the Responses API. Create a `.env` file with the environment variables needed for the API.

If you are using OpenAI, check your [OpenAI account](https://platform.openai.com/api-keys) for creating an API key. Your `.env` file will look like this:

```sh
$ cat .env
OPENAI_API_KEY='sk-YourOpenAiApiKeyHere'
```

If you are using Microsoft Azure OpenAI:
- Go to [Azure Portal](https://portal.azure.com/) > **All Resources**
- Filter the list with Type == Azure OpenAI
- Select the one you plan to use
- If there are none, you can [create and deploy an Azure OpenAI Service resource](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/create-resource?pivots=web-portal#create-a-resource)
- Click on **Keys and Endpoint** on the left menu
- Get `AZURE_OPENAI_API_KEY` and `AZURE_OPENAI_ENDPOINT`
- Click on **Model deployments**, and get the deployment name for `AZURE_OPENAI_DEPLOYMENT_ID`

```sh
$ cat .env
AZURE_OPENAI_API_KEY='yourAzureOpenAiApiKey'
AZURE_OPENAI_ENDPOINT='https://your-azure-deployment.openai.azure.com/'
AZURE_OPENAI_DEPLOYMENT_ID='your-deployment-name'
AZURE_OPENAI_API_VERSION='2025-04-01-preview'
```

One caveat if you are on Azure: the Responses API is supported there, across most regions, but a few pieces lag behind. Multi-turn image editing with streaming is not supported, and background mode with streaming has known performance issues. Check the [Azure Responses API page](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses) before you depend on a specific feature.

In [4]:
# Upload or create a .env file with (Azure) OpenAI API creds

#!echo "OPENAI_API_KEY=sk-YourOpenApiKeyHere" >> .env

#!echo "AZURE_OPENAI_API_KEY=yourAzureOpenAiApiKey" >> .env
#!echo "AZURE_OPENAI_ENDPOINT=https://your-azure-deployment.openai.azure.com/" >> .env
#!echo "AZURE_OPENAI_DEPLOYMENT_ID=your-deployment-name" >> .env
#!echo "AZURE_OPENAI_API_VERSION=2025-04-01-preview" >> .env

### Load `.env` File and Specify (Azure) OpenAI Model

Load environment variables from the `.env` file:

In [5]:
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

Set `IS_AZURE_OPENAI` flag to `True`, if you are using Azure OpenAI:

In [6]:
IS_AZURE_OPENAI: bool = False

Specify the model. `gpt-5.4-nano` is the cheapest model in the GPT-5.4 family, and it supports everything this notebook demonstrates, including reasoning and built-in tools. Swap in a larger model if you want better answers, but the API shapes shown here do not change.

In [7]:
OPENAI_MODEL: str = "gpt-5.4-nano"

---

## Create the Client

Both APIs live on the same client object. `client.chat.completions` calls `/v1/chat/completions`, and `client.responses` calls `/v1/responses`. Nothing about authentication or configuration changes between them, which is the first useful fact: porting is a call-site change, not a client change.

In [8]:
import json
import os
from openai import OpenAI, AzureOpenAI, BadRequestError


def create_openai_client():
    if IS_AZURE_OPENAI:
        return AzureOpenAI(
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_ID"),
        )
    return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


client = create_openai_client()
model = os.getenv("AZURE_OPENAI_DEPLOYMENT_ID") if IS_AZURE_OPENAI else OPENAI_MODEL

print(f"Client: {type(client).__name__}, model: {model}")

Client: OpenAI, model: gpt-5.4-nano


A small helper to keep the output of later cells readable:

In [9]:
def show(title: str, obj, limit: int = 1600):
    """Pretty-print a dict, Pydantic model, or list of them, under a heading."""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    elif isinstance(obj, list):
        obj = [o.model_dump() if hasattr(o, "model_dump") else o for o in obj]

    text = json.dumps(obj, indent=2, default=str)
    print(f"--- {title} ---")
    print(text if len(text) <= limit else text[:limit] + f"\n... [{len(text) - limit} more chars]")
    print()

---

## The Same Call, Both Ways

Start with the smallest possible comparison: one system instruction, one user question, one answer.

### Chat Completions

Everything goes into a single `messages` array. The system prompt is the first element, tagged with a role. The answer comes back at `choices[0].message.content`. The `choices` list exists because of the `n` parameter, which could ask for several independent completions in one call.

In [10]:
chat_response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "Answer in one short sentence."},
        {"role": "user", "content": "What is the capital of France?"},
    ],
)

print(chat_response.choices[0].message.content)

The capital of France is Paris.


### Responses

Three things moved:

- The system prompt became a top-level `instructions` parameter, separate from the conversation.
- The user turn became `input`, which accepts a bare string for the simple case, or a list of items for everything else.
- The answer is at `output_text`, a convenience property the SDK computes for you.

In [11]:
response = client.responses.create(
    model=model,
    instructions="Answer in one short sentence.",
    input="What is the capital of France?",
)

print(response.output_text)

The capital of France is Paris.


Same model, same answer, different envelope. The `instructions` split is not cosmetic. Because it sits outside the conversation, it is not carried forward when you chain turns with `previous_response_id`, so you set it fresh on each call. That is a deliberate design: system-level behavior belongs to the request, not to the transcript. It also means an instruction you set on turn one silently stops applying on turn two unless you set it again, which is the single most common surprise when porting.

You can still put a system or developer message inside `input` as an item if you prefer the old behavior of instructions being part of the replayed history.

### Request Parameter Mapping

The renames you will hit on the first pass:

| Chat Completions | Responses | Note |
|---|---|---|
| `messages=[...]` | `input=` (string or item list) | A bare string is legal for a single user turn |
| `{"role": "system", ...}` in `messages` | `instructions=` (top level) | Not carried across chained turns |
| `max_tokens` / `max_completion_tokens` | `max_output_tokens` | Same meaning, third name for it |
| `response_format={...}` | `text={"format": {...}}` | Structured output moved one level down |
| `reasoning_effort="high"` | `reasoning={"effort": "high"}` | Now an object, with `summary` and `context` alongside |
| `verbosity="low"` | `text={"verbosity": "low"}` | Moved in beside the format config |
| `tools=[{"type": "function", "function": {...}}]` | `tools=[{"type": "function", ...}]` | The nested `function` wrapper is gone |
| `choices[0].message.content` | `output_text` or walk `output[]` | The big one, covered next |

---

## The Output: Items, Not Choices

`output_text` is a convenience. It concatenates the text parts of the output and hides the structure underneath. That structure is the actual change, and you have to understand it before you write anything beyond a demo.

Chat Completions returns a fixed shape: a list of choices, each with exactly one message, and everything the model did is a field on that message. Text is in `content`, tool calls are in `tool_calls`, refusals are in `refusal`. The envelope was designed for one thing, chat, and every capability added since has been bolted onto that one message object.

Responses returns a list of typed **items** in `output`. Each item is one thing the model did. A plain answer is one `message` item. An answer that required a search and some thinking is three or four items in the order they happened.

In [12]:
show("Chat Completions: choices[0]", chat_response.choices[0])
show("Responses: output[]", response.output)

--- Chat Completions: choices[0] ---
{
  "finish_reason": "stop",
  "index": 0,
  "logprobs": null,
  "message": {
    "content": "The capital of France is Paris.",
    "refusal": null,
    "role": "assistant",
    "annotations": [],
    "audio": null,
    "function_call": null,
    "tool_calls": null
  }
}

--- Responses: output[] ---
[
  {
    "id": "msg_0aa65d37dfb932b7006a7c8ef7c8a8819486ca688ca720f8d0",
    "content": [
      {
        "annotations": [],
        "text": "The capital of France is Paris.",
        "type": "output_text",
        "logprobs": []
      }
    ],
    "role": "assistant",
    "status": "completed",
    "type": "message",
    "phase": "final_answer"
  }
]



Read the two dumps side by side. The Chat Completions message carries `content`, `refusal`, `tool_calls`, and `annotations` as fields that are mostly null. The Responses item is a `message` whose `content` is a list of parts, each with its own type. There is no null field for a search that did not happen, because a search that did not happen produces no item.

Here is the same idea with a request that actually produces more than one item. Asking a reasoning model to think adds a `reasoning` item ahead of the message.

In [13]:
reasoning_response = client.responses.create(
    model=model,
    input=(
        "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
        "How much does the ball cost?"
    ),
    reasoning={"effort": "high", "summary": "auto"},
)

print("Item types in output:", [item.type for item in reasoning_response.output])
print("Reasoning tokens:", reasoning_response.usage.output_tokens_details.reasoning_tokens)
print()
print(reasoning_response.output_text)

Item types in output: ['reasoning', 'message']
Reasoning tokens: 34

Let the ball cost \(x\) dollars.  
Then the bat costs \(x + 1.00\) dollars.

Total cost:
\[
x + (x + 1.00) = 1.10
\]
\[
2x + 1.00 = 1.10
\]
\[
2x = 0.10
\]
\[
x = 0.05
\]

**So, the ball costs \(\$0.05\).**


The reasoning item is the model's own scratchpad. You do not get the raw chain of thought, but with `summary: "auto"` you get a readable summary of it, which is useful for debugging why a model went the way it did.

In [14]:
for item in reasoning_response.output:
    if item.type == "reasoning":
        for part in item.summary:
            print(part.text)

**Calculating ball and bat value**

I'm looking into a classic equation where I have the value of the ball as x and the bat as x+1. So, that gives me a total of 2x + 1, which equals 1.10. Now, I can solve for x, and it seems like x equals 0.05. This means the ball costs 5 cents, and the bat costs 6 cents. Let’s go ahead and provide that information!


### The Full Item Vocabulary

`output` is a union of item types. This many, in the current SDK:

In [15]:
import typing
from openai.types.responses.response_output_item import ResponseOutputItem

item_types = typing.get_args(typing.get_args(ResponseOutputItem)[0])
names = sorted(getattr(t, "__name__", str(t)) for t in item_types)

print(f"{len(names)} output item types:\n")
for name in names:
    print(" ", name)

28 output item types:

  AdditionalTools
  ImageGenerationCall
  LocalShellCall
  LocalShellCallOutput
  McpApprovalRequest
  McpApprovalResponse
  McpCall
  McpListTools
  Program
  ProgramOutput
  ResponseApplyPatchToolCall
  ResponseApplyPatchToolCallOutput
  ResponseCodeInterpreterToolCall
  ResponseCompactionItem
  ResponseComputerToolCall
  ResponseComputerToolCallOutputItem
  ResponseCustomToolCall
  ResponseCustomToolCallOutputItem
  ResponseFileSearchToolCall
  ResponseFunctionShellToolCall
  ResponseFunctionShellToolCallOutput
  ResponseFunctionToolCall
  ResponseFunctionToolCallOutputItem
  ResponseFunctionWebSearch
  ResponseOutputMessage
  ResponseReasoningItem
  ResponseToolSearchCall
  ResponseToolSearchOutputItem


You will never handle all of them. You handle the ones your `tools` config can produce, and you must not assume `output[0]` is the message. The common porting bug is exactly that: code that reads `response.output[0].content[0].text`, works fine in testing, and breaks the day a reasoning item or a tool call lands in front of it.

Write the loop instead:

In [16]:
def extract_text(resp) -> str:
    """Walk output items and collect assistant text. This is what output_text does."""
    parts = []
    for item in resp.output:
        if item.type == "message":
            for content in item.content:
                if content.type == "output_text":
                    parts.append(content.text)
    return "".join(parts)


print("Hand-rolled :", extract_text(reasoning_response)[:80])
print("output_text :", reasoning_response.output_text[:80])
print("Identical   :", extract_text(reasoning_response) == reasoning_response.output_text)

Hand-rolled : Let the ball cost \(x\) dollars.  
Then the bat costs \(x + 1.00\) dollars.

Tot
output_text : Let the ball cost \(x\) dollars.  
Then the bat costs \(x + 1.00\) dollars.

Tot
Identical   : True


### Usage and Token Accounting

The usage object was renamed to match the new vocabulary. Both report the same underlying numbers.

| Chat Completions | Responses |
|---|---|
| `usage.prompt_tokens` | `usage.input_tokens` |
| `usage.completion_tokens` | `usage.output_tokens` |
| `usage.prompt_tokens_details.cached_tokens` | `usage.input_tokens_details.cached_tokens` |
| `usage.completion_tokens_details.reasoning_tokens` | `usage.output_tokens_details.reasoning_tokens` |

Reasoning tokens are billed as output tokens in both. They also consume context window space, which matters once you start chaining turns.

In [17]:
show("Chat Completions usage", chat_response.usage)
show("Responses usage", response.usage)

--- Chat Completions usage ---
{
  "completion_tokens": 10,
  "prompt_tokens": 23,
  "total_tokens": 33,
  "completion_tokens_details": {
    "accepted_prediction_tokens": 0,
    "audio_tokens": 0,
    "reasoning_tokens": 0,
    "rejected_prediction_tokens": 0
  },
  "prompt_tokens_details": {
    "audio_tokens": 0,
    "cache_write_tokens": null,
    "cached_tokens": 0
  }
}

--- Responses usage ---
{
  "input_tokens": 23,
  "input_tokens_details": {
    "cache_write_tokens": 0,
    "cached_tokens": 0
  },
  "output_tokens": 11,
  "output_tokens_details": {
    "reasoning_tokens": 0
  },
  "total_tokens": 34
}



One capability that only exists on the Responses side: counting input tokens before you spend them. `/v1/responses/input_tokens` prices a request server-side, with the exact tokenizer the model uses, without running inference. On Chat Completions you approximate this locally with `tiktoken` and hope your approximation matches.

In [18]:
estimate = client.responses.input_tokens.count(
    model=model,
    instructions="Answer in one short sentence.",
    input="What is the capital of France?",
)

print("Counted before the call :", estimate.input_tokens)
print("Billed on the real call :", response.usage.input_tokens)

Counted before the call : 23
Billed on the real call : 23


---

## Conversation State: Who Holds the Transcript

This is one of the significant changes, and the one most likely to change your architecture.

Chat Completions is stateless. The server remembers nothing between calls. A ten-turn conversation means you keep the growing array in your application, and on turn ten you send all twenty messages again. That is either annoying or liberating depending on what you are building.

In [19]:
# Chat Completions: your application owns the array and grows it.

conversation = [{"role": "user", "content": "What is the capital of France? One word."}]

turn1 = client.chat.completions.create(model=model, messages=conversation)
conversation.append(turn1.choices[0].message.model_dump(exclude_none=True))
print("Turn 1:", turn1.choices[0].message.content)

conversation.append({"role": "user", "content": "What is its population? One short sentence."})
turn2 = client.chat.completions.create(model=model, messages=conversation)
conversation.append(turn2.choices[0].message.model_dump(exclude_none=True))
print("Turn 2:", turn2.choices[0].message.content)

print(f"\nMessages your code now holds and resends every turn: {len(conversation)}")

Turn 1: Paris
Turn 2: Paris has a population of about 2.1 million people (within the city limits).

Messages your code now holds and resends every turn: 4


### Option 1: Chain With `previous_response_id`

Responses can hold the transcript for you. Send the new turn only, plus the ID of the response you are continuing from. The server reconstructs the history.

In [20]:
r1 = client.responses.create(
    model=model,
    input="What is the capital of France? One word.",
)
print("Turn 1:", r1.output_text)

r2 = client.responses.create(
    model=model,
    input="What is its population? One short sentence.",
    previous_response_id=r1.id,
)
print("Turn 2:", r2.output_text)

r3 = client.responses.create(
    model=model,
    input="And the river that runs through it? One word.",
    previous_response_id=r2.id,
)
print("Turn 3:", r3.output_text)

Turn 1: Paris
Turn 2: Paris has a population of about 2.1 million people.
Turn 3: Seine


Your code sent three short strings. The model saw the whole conversation. You can ask the server exactly what it replayed:

In [21]:
replayed = client.responses.input_items.list(r3.id)

print(f"Items the server prepended to turn 3: {len(replayed.data)}\n")
for item in reversed(replayed.data):
    text = ""
    if getattr(item, "content", None):
        text = getattr(item.content[0], "text", "")
    print(f"  {item.type:10s} {getattr(item, 'role', ''):10s} {text[:60]}")

Items the server prepended to turn 3: 5

  message    user       What is the capital of France? One word.
  message    assistant  Paris
  message    user       What is its population? One short sentence.
  message    assistant  Paris has a population of about 2.1 million people.
  message    user       And the river that runs through it? One word.


### The Bill Does Not Shrink

This is the part that trips people up, so measure it rather than assume it. `previous_response_id` moves *transmission* of the history to the server. It does not move *billing*. Every replayed token is charged as an input token on the new call, exactly as if you had sent it yourself.

In [22]:
print(f"{'Turn':<8}{'Input tokens':>14}{'Output tokens':>15}")
print("-" * 37)
for label, resp in [("Turn 1", r1), ("Turn 2", r2), ("Turn 3", r3)]:
    print(f"{label:<8}{resp.usage.input_tokens:>14}{resp.usage.output_tokens:>15}")

print(f"\nInput tokens grew {r3.usage.input_tokens / r1.usage.input_tokens:.1f}x "
      f"across three turns, while each request body stayed one short string.")

Turn      Input tokens  Output tokens
-------------------------------------
Turn 1              16              5
Turn 2              36             17
Turn 3              70              6

Input tokens grew 4.4x across three turns, while each request body stayed one short string.


The input token count climbs turn over turn because the transcript grows, which is exactly what happens on Chat Completions too. What you save is bandwidth and the code that manages the array. What you do not save is money.

There is a real cost argument for the Responses API, but it runs through prompt caching, not through the transcript. Because the server holds a stable, byte-identical prefix and it knows the chain, it caches that prefix far more reliably than a client that rebuilds the array each turn. OpenAI reports 40% to 80% better cache utilization than Chat Completions in internal tests, and cached input tokens bill at roughly 10% of the standard rate. Whether you see that depends on your traffic shape: it needs conversations long enough and frequent enough for the cache to stay warm.

You can help the cache along explicitly, on either API, with `prompt_cache_key`, which routes requests sharing a key to the same cache. Caching needs a prefix of roughly 1,024 tokens before it engages, so a short prompt shows nothing.

In [23]:
# Caching needs a prefix of at least ~1024 tokens, so build a long instruction block.
catalogue = [
    f"SKU-{i:04d}: widget model {i}, weight {i % 17} kg, ships from warehouse {i % 5}."
    for i in range(300)
]
long_instructions = (
    "You are a careful assistant answering questions about a product catalogue. "
    + " ".join(catalogue)
)

first = client.responses.create(
    model=model,
    instructions=long_instructions,
    input="How much does SKU-0042 weigh? Number only.",
    prompt_cache_key="ml4devs-codelab-demo",
)
second = client.responses.create(
    model=model,
    instructions=long_instructions,
    input="And SKU-0043? Number only.",
    prompt_cache_key="ml4devs-codelab-demo",
)

for label, resp in [("First call ", first), ("Second call", second)]:
    detail = resp.usage.input_tokens_details
    print(f"{label}: {resp.usage.input_tokens:>6} input tokens, "
          f"{detail.cached_tokens:>6} of them cached")

First call :   6334 input tokens,      0 of them cached
Second call:   6331 input tokens,   5888 of them cached


That is the mechanism worth understanding. The second call reused most of the prefix at roughly a tenth of the price, and the saving came from the prefix being stable, not from where the transcript was stored. Chat Completions caches too, on the same rules. What the Responses API changes is how reliably the prefix stays byte-identical, because the server assembles it instead of your code rebuilding it each turn.

### Option 2: The Conversations API

`previous_response_id` gives you a chain, and a chain is a linked list you have to walk. If you want a durable object with its own identifier, one you can attach turns to from different processes or devices, use the [Conversations API](https://developers.openai.com/api/docs/api-reference/conversations). Create it once, then pass `conversation=` instead of `previous_response_id=`.

In [24]:
conv = client.conversations.create(metadata={"app": "ml4devs-codelab"})
print("Conversation id:", conv.id)

a = client.responses.create(
    model=model,
    conversation=conv.id,
    input="My name is Satish and I work on data infrastructure. Reply with just 'noted'.",
)
b = client.responses.create(
    model=model,
    conversation=conv.id,
    input="What do I work on? Answer in three words.",
)

print("Turn A:", a.output_text)
print("Turn B:", b.output_text)

Conversation id: conv_6a7c8f04c3ec8190b8cfa3d690887d310fe3e790012484bb
Turn A: noted
Turn B: Data infrastructure systems


In [25]:
items = client.conversations.items.list(conv.id)

print(f"Items stored on the conversation: {len(items.data)}\n")
for item in reversed(items.data):
    text = getattr(item.content[0], "text", "") if getattr(item, "content", None) else ""
    print(f"  {getattr(item, 'role', item.type):10s} {text[:60]}")

Items stored on the conversation: 4

  user       My name is Satish and I work on data infrastructure. Reply w
  assistant  noted
  user       What do I work on? Answer in three words.
  assistant  Data infrastructure systems


The two mechanisms differ in retention, and this is worth reading carefully before you pick one.

- Responses stored with `store=True`, which is the default, are kept for 30 days and then dropped.
- Items attached to a conversation object are **not** subject to that 30-day TTL. They persist until you delete them.

So the Conversations API is not just a nicer chaining API. It is a hosted datastore for user conversations, on OpenAI's infrastructure, with your retention policy now partly delegated to a vendor. That is a compliance question before it is an engineering one.

In [26]:
client.conversations.delete(conv.id)
print("Conversation deleted.")

Conversation deleted.


### Option 3: Stateless Responses

You can use the Responses API and keep the transcript yourself. Set `store=False` and pass the accumulated item list as `input`. This is Chat Completions ergonomics with the Responses envelope, and it is the right default for anyone who cannot store user content on a vendor's servers.

In [27]:
context = [{"role": "user", "content": "What is the capital of France? One word."}]

s1 = client.responses.create(model=model, input=context, store=False)
print("Turn 1:", s1.output_text)

# Append the model's own output items, then the next user turn.
# Pass the SDK item objects straight through; hand-built dicts drop or mangle
# fields the server expects back verbatim.
context += list(s1.output)
context.append({"role": "user", "content": "What is its population? One short sentence."})

s2 = client.responses.create(model=model, input=context, store=False)
print("Turn 2:", s2.output_text)

Turn 1: Paris
Turn 2: Paris has a population of about 2.1 million people.


With `store=False`, nothing is retrievable afterwards. The response ID exists only in your process memory, and the server has no record of it.

In [28]:
try:
    client.responses.retrieve(s1.id)
    print("Retrieved. Unexpected.")
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:140]}")

NotFoundError: Error code: 404 - {'error': {'message': "Response with id 'resp_04d1a211fcc45752016a7c8f090fe0819498effa8201acf8bd' not found.", 'type': 'in


Note the default. Chat Completions requests are not stored unless you opt in with `store=True`. Responses requests **are** stored unless you opt out. Porting a privacy-sensitive workload without changing anything else silently starts retaining user data for 30 days. That one flag has ended more Responses API migrations than any missing feature.

### Growing Context and Compaction

Whoever holds the transcript still has to deal with it outgrowing the context window. Chat Completions gives you nothing here: you truncate, summarize, or crash. Responses gives you three options.

- `truncation="auto"` lets the server drop items from the middle when the window is exceeded, rather than failing the request.
- `context_management=[{"type": "compaction", "compact_threshold": N}]` triggers server-side compaction once the rendered conversation crosses N tokens. The server replaces older items with an opaque `compaction` item that preserves the important state and prior reasoning in fewer tokens.
- `client.responses.compact()` does the same thing on demand.

Compaction items are opaque. You cannot read them, and you should not try to. They are a checkpoint the model understands, not a summary for humans. This is the sort of thing you would otherwise build yourself, badly, in every agent project.

In [29]:
managed = client.responses.create(
    model=model,
    input="In one sentence, what is a context window?",
    truncation="auto",
    context_management=[{"type": "compaction", "compact_threshold": 100_000}],
)

print(managed.output_text)
print("\nStatus:", managed.status)

A context window is the amount of input (text) a model considers at once to understand and generate a response.

Status: completed


---

## Tools: Your Functions, and OpenAI's

Tools split into two categories on the Responses API, and only one of them is genuinely new.

### Your Own Functions: A Shape Change

Function calling works on both APIs. What changed is the JSON shape, in three places.

**Declaration.** Chat Completions uses external tagging: a `type` field naming the kind, and a nested `function` object holding the definition. Responses flattens it: the fields sit directly on the tool object next to `type`.

In [30]:
# Chat Completions: nested under "function"
chat_tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
}]

# Responses: flat
responses_tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "Get the current weather for a city",
    "parameters": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
        "additionalProperties": False,
    },
    "strict": True,
}]

Note `strict` in the Responses version. Responses defaults strict function calling to on, Chat Completions defaults it to off. Strict mode guarantees the arguments validate against your schema, but it also demands a stricter schema: every property listed in `required`, and `additionalProperties: False` on every object. A schema that Chat Completions accepted happily can be rejected after a port. Set `"strict": False` if you want the old permissive behavior back.

**The call the model returns.** On Chat Completions it is a `tool_calls` entry on the message, with the name and arguments nested under `function`, and an `id`. On Responses it is a top-level `function_call` item in `output`, flat, with a `call_id`.

In [31]:
chat_tool_call = client.chat.completions.create(
    model=model,
    tools=chat_tools,
    messages=[{"role": "user", "content": "What is the weather in Bengaluru?"}],
)
show("Chat Completions tool call", chat_tool_call.choices[0].message.tool_calls[0])
print("finish_reason:", chat_tool_call.choices[0].finish_reason)
print()

responses_tool_call = client.responses.create(
    model=model,
    tools=responses_tools,
    input="What is the weather in Bengaluru?",
)
print("Output item types:", [item.type for item in responses_tool_call.output])
show("Responses tool call", [i for i in responses_tool_call.output if i.type == "function_call"][0])

--- Chat Completions tool call ---
{
  "id": "call_2xe4XAzQ72FA02YKvf3FTM5e",
  "function": {
    "arguments": "{\"city\":\"Bengaluru\"}",
    "name": "get_weather"
  },
  "type": "function"
}

finish_reason: tool_calls

Output item types: ['function_call']
--- Responses tool call ---
{
  "arguments": "{\"city\":\"Bengaluru\"}",
  "call_id": "call_A9juDwV3Dp5kKglRte7FeAfM",
  "name": "get_weather",
  "type": "function_call",
  "id": "fc_042b314dbb0e1423006a7c8f10bb508197b89c08bb6092213b",
  "caller": null,
  "namespace": null,
  "status": "completed"
}



Note also that Chat Completions signals the tool call through `finish_reason == "tool_calls"`. Responses has no equivalent flag. You detect a tool call by finding a `function_call` item in `output`, which is the same "iterate the items" discipline as before.

**Returning the result.** Chat Completions wants a message with `role: "tool"` and a `tool_call_id`. Responses wants an item with `type: "function_call_output"` and a `call_id`. Getting this field name wrong is the most common single error in a port.

In [32]:
def get_weather(city: str) -> dict:
    """Stand-in for a real weather API."""
    return {"city": city, "temp_c": 27, "sky": "cloudy"}

In [33]:
# Chat Completions round trip: append the assistant message, then a "tool" message.
messages = [{"role": "user", "content": "What is the weather in Bengaluru?"}]
messages.append(chat_tool_call.choices[0].message.model_dump(exclude_none=True))

call = chat_tool_call.choices[0].message.tool_calls[0]
messages.append({
    "role": "tool",
    "tool_call_id": call.id,
    "content": json.dumps(get_weather(**json.loads(call.function.arguments))),
})

final_chat = client.chat.completions.create(model=model, tools=chat_tools, messages=messages)
print("Chat Completions:", final_chat.choices[0].message.content)

Chat Completions: In **Bengaluru**, the weather is currently **27°C and cloudy**.


In [34]:
# Responses round trip: append every output item, then a "function_call_output" item.
items = [{"role": "user", "content": "What is the weather in Bengaluru?"}]
items += list(responses_tool_call.output)

fc = [i for i in responses_tool_call.output if i.type == "function_call"][0]
items.append({
    "type": "function_call_output",
    "call_id": fc.call_id,
    "output": json.dumps(get_weather(**json.loads(fc.arguments))),
})

final_response = client.responses.create(model=model, tools=responses_tools, input=items)
print("Responses:", final_response.output_text)

Responses: The weather in **Bengaluru** is currently **27°C and cloudy**.


If you are chaining with `previous_response_id`, the replay gets shorter: send only the `function_call_output` item as `input`, because the server already holds the call.

None of this is hard. It is the mechanical part of a port, and it is the part a compatibility shim absorbs once. What follows is the part that has no Chat Completions equivalent.

### Built-in Tools: The Actual New Capability

Chat Completions can only call functions that run in your process. You write the search client, you host the vector store, you sandbox the code execution, you handle the retries. Responses lets you declare a tool that OpenAI runs on its own infrastructure, inside the model's turn, before the answer comes back.

Here is web search as one line of configuration.

In [35]:
searched = client.responses.create(
    model=model,
    tools=[{"type": "web_search"}],
    input="What is today's date, and one notable AI news headline? Answer in two lines.",
)

print("Output item types:", [item.type for item in searched.output])
print()
print(searched.output_text)

Output item types: ['web_search_call', 'web_search_call', 'message']

Today is **August 12, 2026**.   
Notable AI news headline: **“Anthropic’s text watermarks signal new front in AI detection.”** ([axios.com](https://www.axios.com/2026/08/12/anthropic-claude-watermarks-ai-detection?utm_source=openai))


There was no loop in that cell. The model decided to search, the search ran on OpenAI's side, the results went into its context, and the answer came back citing sources. Every search it ran is visible as a `web_search_call` item, so the behavior is auditable rather than magic.

In [36]:
for item in searched.output:
    if item.type == "web_search_call":
        action = item.model_dump().get("action") or {}
        print(f"  status={item.status}  action={action.get('type')}  query={str(action.get('query'))[:70]}")

  status=completed  action=search  query=notable AI news headline today August 12 2026
  status=completed  action=search  query=August 12 2026 AI news headline


Citations arrive as annotations on the text, with character offsets into the answer, so you can render them as footnotes without parsing the prose.

In [37]:
for item in searched.output:
    if item.type == "message":
        for content in item.content:
            for ann in (content.annotations or []):
                print(f"  [{ann.start_index}:{ann.end_index}] {ann.title[:50]}")
                print(f"      {ann.url[:100]}")

  [127:233] Anthropic's text watermarks signal new front in AI
      https://www.axios.com/2026/08/12/anthropic-claude-watermarks-ai-detection?utm_source=openai


The full hosted tool set, and what each replaces on your side:

| Tool | `type` | What you would otherwise build |
|---|---|---|
| Web search | `web_search` | Search API integration, result ranking, citation plumbing |
| File search | `file_search` | Vector store, chunking, embedding, retrieval |
| Code interpreter | `code_interpreter` | A sandboxed Python container with a file system |
| Image generation | `image_generation` | A second API call and its own plumbing |
| Computer use | `computer_use_preview` | Screenshot loop and input synthesis |
| Remote MCP | `mcp` | An MCP client, and the call loop around it |
| Shell | `shell` / `local_shell` | Command execution and output capture |
| Apply patch | `apply_patch` | File edit application |
| Tool search | `tool_search` | Loading tool definitions on demand to save context |
| Programmatic tool calling | `code` | Orchestration code that composes several tool calls |

These are billed on top of tokens:

- Web search: $10 per 1,000 calls, plus the retrieved content billed at model input rates.
- File search: $2.50 per 1,000 calls, plus $0.10 per GB per day of vector store storage, with the first GB free.
- Code interpreter: $0.03 to $1.92 per 20-minute session, depending on memory size.

Note that a single request can run several tool calls. The cell above made more than one search. Priced per call, an agent that searches aggressively adds up faster than the token line suggests, so meter it.

### The Nuance That Complicates the Story

"Built-in tools are Responses-only" is the usual summary and it is not quite right. Chat Completions has a `web_search_options` parameter, usable with the search-tuned models, and it returns url citations as annotations.

In [38]:
try:
    cc_search = client.chat.completions.create(
        model="gpt-5-search-api",
        web_search_options={},
        messages=[{"role": "user", "content": "One notable AI news headline today, one line."}],
    )
    print(cc_search.choices[0].message.content[:300])
    print()
    for ann in (cc_search.choices[0].message.annotations or [])[:2]:
        print(" ", ann.url_citation.url[:90])
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:200]}")

OpenAI has temporarily paused parts of its development on the “Astra” model after risks of unsafe behavior emerged during scaling. ([axios.com](https://www.axios.com/2026/08/10/ai-fear-factor-openai-anthropic-hack?utm_source=openai))

  https://www.axios.com/2026/08/10/ai-fear-factor-openai-anthropic-hack?utm_source=openai


So web search specifically has a Chat Completions form. What has no equivalent is everything else: file search, code interpreter, computer use, remote MCP servers, and the hosted shell. If your reason for porting is web search alone, check `web_search_options` first. If it is any of the others, Responses is the only door.

---

## Reasoning Tokens Across Turns

Reasoning models produce a hidden chain of thought before they answer. Those reasoning tokens are billed as output tokens and they occupy context window space. The question that matters across a multi-turn conversation is whether the model can still see what it worked out on turn one when it answers turn three.

On Chat Completions the answer is no. The reasoning is discarded once the response is returned. Turn three sees the earlier answers, not the thinking that produced them.

On Responses, reasoning is a first-class item, and there are two ways to carry it forward.

### Stateful: Chaining Carries It for You

With `previous_response_id` or a conversation, reasoning items live server-side alongside the messages, and the model can access them on later turns. `reasoning.context` controls how far back it looks:

- `current_turn`: only the reasoning from the active turn.
- `all_turns`: reasoning from earlier turns too, which is the default on the newest models.

In [39]:
step1 = client.responses.create(
    model=model,
    input=(
        "Three friends split a $90 restaurant bill equally. Ann paid $47, Bob paid $31, "
        "Carol paid $12. Give each person's net balance, where a positive number means "
        "the group owes them. List three lines, name and amount only."
    ),
    reasoning={"effort": "high", "summary": "auto", "context": "all_turns"},
)

print("Item types:", [item.type for item in step1.output])
print("Reasoning tokens:", step1.usage.output_tokens_details.reasoning_tokens)
print()
print(step1.output_text)

Item types: ['reasoning', 'message']
Reasoning tokens: 182

Ann: 17  
Bob: 1  
Carol: -18


In [40]:
step2 = client.responses.create(
    model=model,
    previous_response_id=step1.id,
    input="Now redo it assuming the bill was $120 instead. Balances only.",
    reasoning={"effort": "high", "context": "all_turns"},
)

print(step2.output_text)
print()
print("Input tokens on turn 2:", step2.usage.input_tokens)
print("Reasoning tokens on turn 2:", step2.usage.output_tokens_details.reasoning_tokens)

Ann: 7  
Bob: -9  
Carol: -28

Input tokens on turn 2: 279
Reasoning tokens on turn 2: 37


### Stateless: You Carry the Encrypted Blob

If you cannot let OpenAI store the conversation, you can still keep the reasoning. Set `store=False` and ask for `include=["reasoning.encrypted_content"]`. Each reasoning item then comes back with an `encrypted_content` field: an opaque blob that only OpenAI can decrypt. Pass it back in `input` on the next turn and the model regains its earlier thinking, with nothing readable stored on either side.

In [41]:
context = [{
    "role": "user",
    "content": (
        "Three friends split a $90 restaurant bill equally. Ann paid $47, Bob paid $31, "
        "Carol paid $12. Give each person's net balance, where a positive number means "
        "the group owes them. List three lines, name and amount only."
    ),
}]

enc1 = client.responses.create(
    model=model,
    input=context,
    reasoning={"effort": "high"},
    store=False,
    include=["reasoning.encrypted_content"],
)

for item in enc1.output:
    if item.type == "reasoning":
        blob = item.model_dump().get("encrypted_content")
        print(f"Encrypted reasoning blob: {len(blob)} chars, starts {blob[:40]}...")

print()
print(enc1.output_text)

Encrypted reasoning blob: 1380 chars, starts gAAAAABqfI8qKqPz6DIq27KLwNz9MyRlLD00MnPa...

Ann 17  
Bob 1  
Carol -18


In [42]:
# Replay every output item, encrypted reasoning included, to keep continuity.
context += list(enc1.output)
context.append({"role": "user", "content": "Now redo it assuming the bill was $120 instead. Balances only."})

enc2 = client.responses.create(
    model=model,
    input=context,
    reasoning={"effort": "high"},
    store=False,
    include=["reasoning.encrypted_content"],
)

print(enc2.output_text)

Ann 7  
Bob -9  
Carol -28


The failure mode here is quiet. Drop the reasoning items from your replayed context and everything still works: no error, no warning, just a model that has to re-derive its own earlier thinking, costing you reasoning tokens and sometimes changing the answer. If you replay manually, replay the whole `output` list, not the messages you find interesting.

`reasoning.effort` takes `none`, `minimal`, `low`, `medium`, `high`, `xhigh`, and `max`, depending on the model. `none` is for latency-critical work that gains nothing from thinking. `medium` is the usual default. Chat Completions has `reasoning_effort` as a flat parameter, so effort itself is not the differentiator. Summaries, `context`, and encrypted continuity are.

---

## Structured Output

Both APIs do the same JSON Schema constrained decoding. The parameter moved and the nesting changed.

In [43]:
person_schema = {
    "type": "object",
    "properties": {
        "city": {"type": "string"},
        "country": {"type": "string"},
        "year_built": {"type": "integer"},
    },
    "required": ["city", "country", "year_built"],
    "additionalProperties": False,
}

cc_json = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Where and when was the Eiffel Tower built?"}],
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "landmark", "schema": person_schema, "strict": True},
    },
)
print("Chat Completions:", cc_json.choices[0].message.content)

r_json = client.responses.create(
    model=model,
    input="Where and when was the Eiffel Tower built?",
    text={
        "format": {
            "type": "json_schema",
            "name": "landmark",
            "schema": person_schema,
            "strict": True,
        }
    },
)
print("Responses       :", r_json.output_text)

Chat Completions: {"city":"Paris","country":"France","year_built":1889}
Responses       : {"city":"Paris","country":"France","year_built":1889}


Two differences worth noting. The schema `name` sits one level higher on Responses, alongside `type`, rather than inside a `json_schema` wrapper. And `text` is also where `verbosity` lives, which Chat Completions exposes as a top-level parameter.

Both SDKs also offer a `parse()` helper that takes a Pydantic model and returns typed objects, `client.chat.completions.parse()` and `client.responses.parse()`. That layer is identical in spirit on both.

---

## Streaming: Deltas vs Semantic Events

Chat Completions streams chunks that all look alike. Each one carries a `delta` and you concatenate `delta.content` until `finish_reason` arrives. If the model calls a tool mid-stream, you reassemble the arguments from partial JSON fragments yourself.

In [44]:
chunks = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Count from 1 to 5, comma separated."}],
    stream=True,
)

pieces, kinds = [], []
for chunk in chunks:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta.content:
        pieces.append(delta.content)
    kinds.append("content" if delta.content else f"finish={chunk.choices[0].finish_reason}")

print("Text  :", "".join(pieces))
print("Chunks:", len(kinds), "| kinds:", sorted(set(kinds)))

Text  : 1, 2, 3, 4, 5
Chunks: 15 | kinds: ['content', 'finish=None', 'finish=stop']


Responses streams typed events. Each one names what just happened, so you can drive a UI off the event type instead of inferring state from the payload.

In [45]:
event_types = []
with client.responses.stream(
    model=model,
    input="Count from 1 to 5, comma separated.",
) as stream:
    for event in stream:
        event_types.append(event.type)
    final = stream.get_final_response()

print("Text  :", final.output_text)
print("Events:", len(event_types))
print()
for name in sorted(set(event_types)):
    print(f"  {name:45s} x{event_types.count(name)}")

Text  : 1, 2, 3, 4, 5
Events: 21

  response.completed                            x1
  response.content_part.added                   x1
  response.content_part.done                    x1
  response.created                              x1
  response.in_progress                          x1
  response.output_item.added                    x1
  response.output_item.done                     x1
  response.output_text.delta                    x13
  response.output_text.done                     x1


`response.created`, `response.output_item.added`, `response.output_text.delta`, `response.output_item.done`, `response.completed`, in that order. When tools are in play you also get `response.web_search_call.in_progress`, `response.function_call_arguments.delta`, `response.reasoning_summary_text.delta`, and so on. The current SDK models more than 50 distinct event types.

This is the difference between rendering a spinner and rendering "searching the web for X". The information was never in the Chat Completions stream to begin with.

In [46]:
from openai.types.responses.response_stream_event import ResponseStreamEvent

stream_events = typing.get_args(ResponseStreamEvent)
if len(stream_events) == 2 and hasattr(stream_events[0], "__args__"):
    stream_events = typing.get_args(stream_events[0])

print(f"Streaming event classes in the SDK: {len(stream_events)}")

Streaming event classes in the SDK: 53


---

## Background Mode

Long-running work has no answer on Chat Completions. A request that takes six minutes holds an HTTP connection for six minutes, and if the connection drops you have paid for tokens you will never see.

Responses can run a request as a background job. You get an ID immediately, poll for status, and retrieve the result when it lands. This is what makes deep research and long agent runs practical.

In [47]:
import time

job = client.responses.create(
    model=model,
    input="Write a haiku about idempotent APIs.",
    background=True,
)
print("Immediately:", job.status, job.id)

for _ in range(60):
    job = client.responses.retrieve(job.id)
    if job.status in ("completed", "failed", "cancelled", "incomplete"):
        break
    time.sleep(1)

print("Final     :", job.status)
print()
print(job.output_text)

Immediately: queued resp_0cb520960e25eea7006a7c8f3050bc8190af8e24242f40c014
Final     : completed

Same request, again—  
answers don’t change with time,  
state stays quietly.


Background responses can also be cancelled with `client.responses.cancel(id)`, and streamed with a resumable cursor if the client reconnects.

---

## What You Give Up

Everything so far has been what Responses adds. Now the other direction. Rather than quote a table that goes stale, compute the parameter difference from the installed SDK.

In [48]:
import inspect
from openai.resources.responses import Responses
from openai.resources.chat.completions import Completions

INFRA = {"self", "extra_headers", "extra_query", "extra_body", "timeout"}

responses_params = set(inspect.signature(Responses.create).parameters) - INFRA
chat_params = set(inspect.signature(Completions.create).parameters) - INFRA

print("Only in Chat Completions:")
for p in sorted(chat_params - responses_params):
    print("  ", p)

print("\nOnly in Responses:")
for p in sorted(responses_params - chat_params):
    print("  ", p)

Only in Chat Completions:
   audio
   frequency_penalty
   function_call
   functions
   logit_bias
   logprobs
   max_completion_tokens
   max_tokens
   messages
   modalities
   n
   prediction
   presence_penalty
   reasoning_effort
   response_format
   seed
   stop
   verbosity
   web_search_options

Only in Responses:
   background
   context_management
   conversation
   include
   input
   instructions
   max_output_tokens
   max_tool_calls
   previous_response_id
   prompt
   reasoning
   text
   truncation


Strip out the pure renames (`messages`/`input`, `max_tokens`/`max_output_tokens`, `response_format`/`text`, `reasoning_effort`/`reasoning`, `verbosity`/`text`) and a real list of losses remains. Confirm they are genuine server-side rejections rather than SDK typing, by pushing them through `extra_body`:

In [49]:
for param, value in [
    ("n", 2),
    ("stop", ["\n"]),
    ("seed", 42),
    ("logit_bias", {"1234": 10}),
    ("presence_penalty", 0.5),
]:
    try:
        client.responses.create(model=model, input="hi", extra_body={param: value})
        print(f"  {param:18s} accepted by the server")
    except BadRequestError as e:
        message = (e.body or {}).get("message", str(e))
        print(f"  {param:18s} {message[:90]}")

  n                  Unknown parameter: 'n'.
  stop               Unknown parameter: 'stop'. Did you mean 'store'?
  seed               Unknown parameter: 'seed'.
  logit_bias         Unknown parameter: 'logit_bias'.
  presence_penalty   accepted by the server


Note the last row. `n`, `stop`, `seed`, and `logit_bias` are hard rejections, but `presence_penalty` sails through and is ignored. The SDK will not let you pass it, so you only hit this through a raw HTTP call or a gateway that forwards unknown fields. Silent acceptance of a parameter that does nothing is worse than a 400, so do not assume a request that succeeds means every knob in it was honored.

What that costs you, in order of how often it actually bites:

| Lost | Why it matters |
|---|---|
| `n` | No multi-sample generation in one call. Self-consistency, best-of-N reranking, and candidate generation now cost N round trips. |
| `stop` | No custom stop sequences. If you parse a delimiter out of the output, you now stop in post-processing and pay for the tokens past it. |
| `seed` | No reproducibility hint. It was best-effort on Chat Completions, but evaluation harnesses and regression tests leaned on it. |
| `logit_bias` | No token-level steering. Banning or forcing specific tokens was the classic trick for constrained classification. |
| `logprobs` | `top_logprobs` exists on Responses, but the flat `logprobs` toggle does not. Confidence scoring and calibration pipelines need reworking. |
| `frequency_penalty`, `presence_penalty` | Repetition control has to move into the prompt. |
| `audio`, `modalities` | Audio in and out is not on this endpoint. Voice work uses the realtime and audio APIs instead. |
| `prediction` | No predicted outputs, which sped up mostly-unchanged rewrites such as code edits. |

For most chat and agent applications none of these matter. For evaluation harnesses, classifiers, and anything that samples multiple candidates, `n`, `seed`, `logprobs`, and `logit_bias` together are a serious loss, and that class of workload is a legitimate reason to stay put.

---

## Portability: The Argument That Usually Decides It

The technical comparison rarely settles the question on its own. This does.

Chat Completions is not just an OpenAI endpoint. It became the de facto wire format for the whole ecosystem. If your code speaks `/v1/chat/completions`, you can point it at:

- Local servers: llama.cpp, Ollama, LM Studio, vLLM, SGLang, TGI
- Gateways and routers: LiteLLM, OpenRouter, Portkey, Cloudflare AI Gateway
- Other vendors' compatibility layers: Together, Fireworks, Groq, Mistral, DeepSeek, Anthropic and Google via shims
- Azure OpenAI, on the same shape

That is what "portable" means in practice: swapping a base URL and a model name, not rewriting a call site.

Responses support outside OpenAI is real but uneven, and that gap is the whole risk.

| Runtime | `/v1/responses` support |
|---|---|
| Azure OpenAI | Supported broadly, with gaps in image generation and background streaming |
| vLLM | Implemented, with multi-turn and hosted-tool behavior still maturing |
| LiteLLM | Proxies `/responses` through, and can bridge to `/chat/completions` for backends that only speak the old shape |
| Ollama, llama.cpp, LM Studio | Chat Completions shape; no native Responses endpoint |
| Third-party model vendors | Almost universally Chat Completions |

And there is a category error hiding in the comparison. Even where `/v1/responses` is implemented elsewhere, the features worth porting for do not come with it. `previous_response_id` requires server-side conversation storage. `web_search` requires OpenAI running a search. A local vLLM server speaking the Responses shape gives you the envelope, not the capabilities. The envelope was never the reason to port.

So the portability question is not "can another server accept this JSON". It is "is the thing I am porting for something another server could ever provide". For hosted state and hosted tools, the answer is no by construction, and you are choosing lock-in deliberately, which is fine as long as you are choosing it deliberately.

---

## Keeping One Call Site

If you are undecided, or you expect to run both, do not scatter either API's shape through your codebase. Put both behind one function and let the rest of your code stay ignorant. This is the same discipline that makes swapping a local model for a hosted one a config change rather than a refactor.

In [50]:
from typing import Literal


def complete(
    prompt: str,
    system: str | None = None,
    api: Literal["responses", "chat"] = "responses",
    state_id: str | None = None,
) -> dict:
    """One call site over both OpenAI surfaces.

    Returns a dict with the text, the token usage, and a state handle that the
    caller passes back on the next turn. On Responses that handle is a response
    id held by the server; on Chat Completions it is the message array your own
    process holds. Callers never need to know which.
    """
    if api == "responses":
        resp = client.responses.create(
            model=model,
            instructions=system,
            input=prompt,
            previous_response_id=state_id,
        )
        return {
            "text": resp.output_text,
            "input_tokens": resp.usage.input_tokens,
            "output_tokens": resp.usage.output_tokens,
            "state": resp.id,
        }

    messages = list(state_id or [])
    if not messages and system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    resp = client.chat.completions.create(model=model, messages=messages)
    messages.append(resp.choices[0].message.model_dump(exclude_none=True))
    return {
        "text": resp.choices[0].message.content,
        "input_tokens": resp.usage.prompt_tokens,
        "output_tokens": resp.usage.completion_tokens,
        "state": messages,
    }

In [51]:
for api in ("responses", "chat"):
    first = complete("Name one Indian classical raga. One word.", system="Be terse.", api=api)
    second = complete("Which time of day is it traditionally sung? Short answer.",
                      api=api, state_id=first["state"])
    print(f"[{api:9s}] {first['text']}  ->  {second['text']}")
    print(f"{'':12s}tokens in/out across both turns: "
          f"{first['input_tokens'] + second['input_tokens']}/"
          f"{first['output_tokens'] + second['output_tokens']}")

[responses] Bhairav  ->  Morning (early morning).
            tokens in/out across both turns: 64/16
[chat     ] Yaman  ->  Evening.
            tokens in/out across both turns: 70/11


The same two turns, the same model, two different transports, one interface. With that shim in place, porting stops being a migration and becomes a flag. That is the position worth being in before you decide, because it makes the decision cheap and reversible.

---

## How to Choose

Work down this list and stop at the first row that matches your situation.

| If | Then | Because |
|---|---|---|
| You need file search, code interpreter, computer use, or remote MCP | **Responses** | There is no Chat Completions equivalent. Building these yourself is weeks of work you would then own. |
| You are building an agent that runs for minutes | **Responses** | Background mode, compaction, and typed streaming events exist only here. |
| You want conversation state you do not host | **Responses** | `previous_response_id` and the Conversations API remove a datastore from your system. |
| You run reasoning models over multi-turn conversations | **Responses** | Reasoning continuity across turns, stateful or encrypted, is Responses-only. |
| You must be able to swap in a local or third-party model | **Chat Completions** | It is what open-weight servers, gateways, and other vendors speak. |
| Your workload needs `n`, `seed`, `logprobs`, or `logit_bias` | **Chat Completions** | Evaluation, sampling, and calibration pipelines depend on parameters Responses dropped. |
| You cannot retain user content on a vendor's servers | **Either, carefully** | Responses works with `store=False`, but the default is retention, so the port is not a no-op. |
| Your app is a single-turn call that works today | **Chat Completions** | Nothing is being taken away. Port when you need something, not on principle. |

### The Short Version

Port when you want hosted state or built-in tools. Stay on Chat Completions when portability across providers matters more, because the Chat Completions shape is what most third-party gateways and open-weight servers speak.

Everything else is detail.

### What Is Not a Reason to Port

- **Deprecation fear.** Chat Completions is not deprecated. The Assistants API is, with a sunset of 26 August 2026, and Responses is its replacement. If you are on Assistants, you are migrating; if you are on Chat Completions, you are choosing.
- **Cache savings alone.** The 40% to 80% cache utilization improvement is real but conditional on conversation shape and traffic volume. Measure it on your own traffic before you build a business case on it.
- **Being current.** A working single-turn call gains nothing from the new envelope.

---

## Summary and Key Takeaways

This notebook ran both APIs against the same model and compared them where the differences are load-bearing.

### What Moved

| Concern | Chat Completions | Responses |
|---|---|---|
| Input | `messages=[...]` | `input=` plus top-level `instructions=` |
| Output | `choices[0].message` | `output[]`, a list of typed items |
| Convenience accessor | `choices[0].message.content` | `output_text` |
| Conversation state | Your array, resent each turn | `previous_response_id`, conversations, or your array |
| Default retention | Not stored unless `store=True` | Stored 30 days unless `store=False` |
| Tool declaration | Nested under `function` | Flat, `strict` on by default |
| Tool result | `role: "tool"` + `tool_call_id` | `function_call_output` + `call_id` |
| Hosted tools | Web search only, via `web_search_options` | Ten tool types, including file search and code interpreter |
| Reasoning across turns | Discarded | Carried, server-side or as encrypted blobs |
| Structured output | `response_format` | `text.format` |
| Streaming | Uniform `delta` chunks | 50+ typed semantic events |
| Long jobs | Hold the connection | `background=True` and poll |
| Token counting | Local estimate | `responses.input_tokens.count()` |

### Facts Worth Remembering

- **Hosted state does not cut your bill.** Replayed history is billed as input tokens either way. The measured token growth in this notebook shows it plainly. Savings come from prompt caching, which is a different mechanism with its own conditions.
- **`store` flips default on the port.** Responses retains for 30 days unless told otherwise, and conversation items have no TTL at all. That is a compliance decision, not a config detail.
- **`instructions` does not persist across chained turns.** Set it on every call.
- **Never index into `output`.** Iterate items by type. Reasoning items and tool calls will appear in front of the message.
- **Replay every output item or lose the reasoning.** Dropping reasoning items from a manual replay fails silently and costs you tokens.
- **Built-in tools are billed per call.** Web search at $10 per 1,000 calls, and one request can make several.
- **Portability is the real trade.** The Chat Completions shape is the ecosystem's lingua franca, and the features worth porting for cannot follow you off OpenAI anyway.

### Where to Go Next

- [Build Your First LLM Application with the OpenAI API](https://www.ml4devs.com/articles/llm-first-app-with-openai-api/) if you want the zero-to-first-call version of this material.
- [Run LLMs Locally](https://www.ml4devs.com/articles/llm-local-inference-dev-setup/) for the other side of the portability argument, running open-weight models behind the same Chat Completions shape.
- [LLM-Powered Text-to-SQL](https://www.ml4devs.com/articles/text-to-sql-with-llm-rag-using-gpt-and-chromadb/) for a full RAG pipeline built on Chat Completions, and a concrete thing to try porting.

---
<p>Copyright &copy 2026 <a href="https://www.linkedin.com/in/scgupta">Satish Chandra Gupta</a>.</p>
<img src="https://licensebuttons.net/l/by-nc-sa/3.0/88x31.png" align="left"/> <p>&nbsp;<a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">CC BY-NC-SA 4.0 International</a> License.</p>